# Modeling

From our EDA, we've made the following hypotheses/statements which we will test and verify:

- Linear regression can be used as a baseline model which will be compared to the improvements we create
- Decision Tree based models using either Boosting or Random Forests will generate the most accurate results 

We decided not to use a neural network (RNN) due to the limited number of labels available. In fact, any deep-learning style approach will most likely result in overfitting on the training data due to the limited sample size of the I-94 arrival data.

## Baseline Model: Linear Regression

### Import Libraries and Load Data

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

The filepaths below use local filepaths: if running on a local machine be sure to change them as required

In [2]:
df_i94     = pd.read_csv("/Users/abbykreutz/github/MIDS207_FinalProject/Peter_EDA/label_arrivals_cleaned.csv")  # Label: total visitors
df_party   = pd.read_csv("/Users/abbykreutz/github/MIDS207_FinalProject/Peter_EDA/features_party_cleaned.csv")  # Feature: Political control
df_tourism = pd.read_csv("/Users/abbykreutz/github/MIDS207_FinalProject/Hope_EDA/features_inbound_cleaned.csv") # Feature: Inbound Tourism
df_gdp     = pd.read_csv("/Users/abbykreutz/github/MIDS207_FinalProject/Hope_EDA/features_gdp_cleaned.csv")     # Feature: Tourism % GDP
df_acled   = pd.read_csv("/Users/abbykreutz/github/MIDS207_FinalProject/Abby_EDA/features_acled_cleaned.csv")   # Feature: Armed conflict data
df_acledv2 = pd.read_csv("/Users/abbykreutz/github/MIDS207_FinalProject/Abby_EDA/features_acled_extended.csv")  # Additional ACLED Feature

### Feature Engineering

First we'll look at political features from the political party in control dataset. We want to include transitional features which gave more insight than simply which party was in power at a given time

In [3]:
# Map each year to the corresponding Congress using start_year
df_party["start_year"] = df_party["years"].str[:4].astype(int)
df_party["end_year"]   = df_party["years"].str[-4:].astype(int)

def get_congress_row(year, df_party):
    mask = (df_party["start_year"] <= year) & (df_party["end_year"] > year)
    return df_party[mask].iloc[0] if mask.any() else None

# Binary encode political features
df_party["is_rep_house"]    = df_party["house_majority"].eq("Republicans").astype(int)
df_party["is_rep_senate"]   = df_party["senate_majority"].str.startswith("Republicans").astype(int)
df_party["is_rep_president"]= df_party["presidency"].str.startswith("Republican").astype(int)
df_party["is_unified"]      = df_party["party_government"].eq("Unified").astype(int)

# Transition features (did control change vs previous Congress?)
df_party["house_changed"]   = (df_party["is_rep_house"] != df_party["is_rep_house"].shift()).astype(int)
df_party["senate_changed"]  = (df_party["is_rep_senate"] != df_party["is_rep_senate"].shift()).astype(int)
df_party["pres_changed"]    = (df_party["is_rep_president"] != df_party["is_rep_president"].shift()).astype(int)

political_features = [
    "is_rep_house", "is_rep_senate", "is_rep_president",
    "is_unified", "house_changed", "senate_changed",
    "pres_changed"
]

# Expand each two-year congressional period into one row per calendar year

party_yearly = (df_party[political_features     # Select all political_features.
    + ["start_year", "end_year"]].assign(       # Plus start_year and end_year columns,
        year=lambda x: x.apply(                 # and a new column, year, that we'll populate.
            lambda row: list(range(             # We create a list of years ranging from
                int(row["start_year"]),         # the start_year of the congressional period
                int(row["end_year"])            # through the end_year (exclusive, so as not to double count!)
                )
            ),
        axis=1                                  # lambda function is applied row-by-row across columns
        )
    )
    .explode("year")                            # transform list of years into dataframe with political feature flags
)

party_yearly["year"] = party_yearly["year"].astype(int)
party_yearly = party_yearly[political_features + ["year"]].set_index("year")

# Each calendar year should map to exactly one Congress
assert not party_yearly.index.duplicated().any()

In [4]:
df_party.head()
party_yearly.head()

,is_rep_house,is_rep_senate,is_rep_president,is_unified,house_changed,senate_changed,pres_changed
year,,,,,,,
1857,0,0,0,1,1,1,1
1858,0,0,0,1,1,1,1
1859,1,0,0,0,1,0,0
1860,1,0,0,0,1,0,0
1861,1,1,1,1,0,1,1


df_acled is already cleaned for usage (thanks Abby!). Next we'll look at election years as another feature: 

In [5]:
# Presidential election years (every 4 years)
df_i94["is_election_year"] = df_i94["year"].isin(
    range(2000, 2027, 4)                                 
).astype(int)

Merge all features into one dataset and filter from 2000-2026 (to match the I-94 dataset):

In [6]:
# ── Leakage fix: tourism and GDP are ANNUAL, and merging them on `year` broadcasts
# the full-year figure onto every month of that year. Predicting Jan 2023 with the
# 2023 annual tourism total means the model sees Feb-Dec 2023 before it has happened.
# Shifting the year forward by 1 gives each month the last COMPLETED year's figure,
# which is genuinely knowable at prediction time.
df_tourism_lagged = df_tourism.assign(year=df_tourism["year"] + 1)
df_gdp_lagged     = df_gdp.assign(year=df_gdp["year"] + 1)

# Political data is at Congress (2-year) level — merge on year
df = df_i94.merge(party_yearly,on="year", how="left", validate="many_to_one")

# Tourism and ACLED are both presented annually, therefore we'll merge on the year
df = df.merge(df_tourism_lagged, on="year", how="left")
df = df.merge(df_gdp_lagged, on="year", how="left")
df = df.merge(df_acled, on="date", how="left")

# Filter the years beteen 2000-2026 and drop all NaN values
df = df[(df["year"] >= 2000) & (df["year"] <= 2026)].reset_index(drop=True)
df = df.dropna()

Now we'll define our features and label constants:

In [7]:
LABEL = "total_arrivals"

FEATURES = [
    # Political
    "is_rep_house", "is_rep_senate", "is_rep_president",
    "is_unified", "house_changed", "senate_changed",
    "pres_changed", "is_election_year",
    # Tourism
    "tourism_value", "tourism_pct_change", "gdp_value", "gdp_pct_change",
    # ACLED
    "acled_incidents",
    "acled_fatal_events",
    "acled_total_fatalities",
    "acled_any_event",
    "acled_any_fatal_event",
    "acled_high_activity_flag",
    "acled_events_lag_1",
    "acled_fatal_events_lag_1",
    "acled_events_lag_2",
    "acled_fatal_events_lag_2",
    "acled_events_roll_3",
    "acled_fatalities_roll_3",
    "acled_states",
    "acled_event_types",
]

FEATURES = [col for col in FEATURES if col in df.columns]

X = df[FEATURES]
y = df[LABEL]

### Train/Test Split and Feature Scaling 

Because we are using chronological data, we will NOT be using random splitting for our data. Instead, we need to delineate a cutoff between the two datasets. In this case, we will use Jan 2022

In [8]:
split_date = "2022-01-01"
train_mask = df["date"] < split_date

X_train, X_test = X[train_mask], X[~train_mask]
y_train, y_test = y[train_mask], y[~train_mask]

print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")

Train size: 24 | Test size: 36


In [9]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  
X_test_scaled  = scaler.transform(X_test)

### Baseline 1: Predict the number of visitors using the average

This first baseline is very simple: calculate and predict the mean number of visitors from y_train every time.

In [10]:
avg_baseline = y_train.mean()
avg_baseline_pred = np.full(len(y_test), avg_baseline)

In [11]:
# Evaluate against actual test-period arrivals
rmse = np.sqrt(mean_squared_error(y_test, avg_baseline_pred))
mae = mean_absolute_error(y_test, avg_baseline_pred)
r2 = r2_score(y_test, avg_baseline_pred)

print(f"Mean Baseline — predicting {avg_baseline:,.0f} arrivals every month\n")
print(f"RMSE : {rmse:>12,.0f}")
print(f"MAE  : {mae:>12,.0f}")
print(f"R²   : {r2:>12.4f}")

Mean Baseline — predicting 1,728,838 arrivals every month

RMSE :    3,706,724
MAE  :    3,532,757
R²   :      -9.9095


We see here very high (> 1 million) RMSE and MAE after using a prediction of 4.2 million visitors for every month.

### Baseline 2: Use Linear Regression to predict number of visitors

Our second baseline will use LR to generate predictions with our features. Because we don't anticipate a linear relationship between our features and label, we don't expect high performance. However for the sake of a baseline model, this will more than suffice.

In [12]:
# Baseline no. 2: predict number of visitors using LR 
lr_baseline = LinearRegression()
lr_baseline.fit(X_train_scaled, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [13]:
# Evaluate against actual test-period arrivals
lr_baseline_pred = lr_baseline.predict(X_test_scaled)

rmse = np.sqrt(mean_squared_error(y_test, lr_baseline_pred))
mae  = mean_absolute_error(y_test, lr_baseline_pred)
r2   = r2_score(y_test, lr_baseline_pred)

print(f"RMSE : {rmse:,.0f}")
print(f"MAE  : {mae:,.0f}")
print(f"R²   : {r2:.4f}")

RMSE : 3,267,496
MAE  : 3,083,663
R²   : -7.4773


These values are slightly better than just predicting the average, but can definitely be improved upon.

In [14]:
# Display coefficients for each feature to determine significance
coef_df = pd.DataFrame({
    "feature"    : FEATURES,
    "coefficient": lr_baseline.coef_
}).sort_values("coefficient", ascending=False)

print(coef_df.to_string(index=False))

           feature    coefficient
  is_rep_president   23042.080486
     is_rep_senate   23042.080486
     house_changed   23042.080486
  is_election_year   23042.080486
     tourism_value   23042.080486
tourism_pct_change   23042.080486
    gdp_pct_change   23042.080486
         gdp_value   23042.080486
      is_rep_house       0.000000
        is_unified  -23042.080486
    senate_changed  -23042.080486
      pres_changed  -23042.080486
   acled_incidents -855838.511775


Our coefficients are indicating the model his having a difficult time determining what features are most significant. This is seen with how many repeat coefficients there are across feature. We did not anticipate any of these relationships to be linear, therefore further experimentation will be used to determine what models will be best suited for this data and problem set.

# Abby Modeling Section

In [15]:
df_baseline = df.copy()

# Uses the extended ACLED feature export
df_extended = (df_i94.merge(party_yearly,on='year', how='left', validate='many_to_one')
    .merge(df_tourism_lagged, on='year', how='left')
    .merge(df_gdp_lagged, on='year', how='left')
    .merge(df_acledv2, on='date', how='left')
)

df_extended = df_extended[(df_extended['year'] >= 2000) & (df_extended['year'] <= 2026)].reset_index(drop=True)
df_extended = df_extended.dropna()

LABEL = 'total_arrivals'
ACLED_FEATURES = [c for c in df_acledv2.columns if c != 'date' and c not in FEATURES]

# Sanity check: how much data actually survives the inner-join + dropna?
ext_mask = df_extended['date'] < split_date
n_train_ext = ext_mask.sum()
n_feat_ext  = len(FEATURES + ACLED_FEATURES)

print(f"Baseline rows           : {len(df_baseline)}")
print(f"Rows after merge/filter : {len(df_extended)}")
print(f"Date range              : {df_extended['date'].min()} to {df_extended['date'].max()}")
print(f"Train rows              : {n_train_ext}")
print(f"Test rows               : {(~ext_mask).sum()}")
print(f"Features                : {n_feat_ext}")

# Is the design matrix full rank (are the columns linearly independent)?
rank = np.linalg.matrix_rank(
    StandardScaler().fit_transform(df_extended.loc[ext_mask, FEATURES + ACLED_FEATURES])
)
print(f"\nDesign matrix rank: {rank} features of the {n_feat_ext} columns")
if rank < n_feat_ext:
    print(f"\rare collinear (highly correlated, i.e., duplicate information)!")
print()

results = []
fitted_lr = {}   # keep each comparison's fitted model so we can read the right coefficients

for name, df_frame, cols in [
    ('baseline_acled', df_baseline, FEATURES),
    ('extended_acled', df_extended, FEATURES + ACLED_FEATURES),
]:
    X = df_frame[cols]
    y = df_frame[LABEL]
    mask = df_frame['date'] < split_date
    X_train, X_test = X[mask], X[~mask]
    y_train, y_test = y[mask], y[~mask]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    lr = LinearRegression().fit(X_train_scaled, y_train)
    pred_lr = lr.predict(X_test_scaled)
    fitted_lr[name] = (lr, cols)
    results.append({
        'comparison': name,
        'model': 'Linear Regression',
        'RMSE': round(np.sqrt(mean_squared_error(y_test, pred_lr)), 2),
        'MAE': round(mean_absolute_error(y_test, pred_lr), 2),
        'R2': round(r2_score(y_test, pred_lr), 4),
    })

    rf = RandomForestRegressor(n_estimators=300, random_state=42)
    rf.fit(X_train, y_train)
    pred_rf = rf.predict(X_test)
    results.append({
        'comparison': name,
        'model': 'Random Forest',
        'RMSE': round(np.sqrt(mean_squared_error(y_test, pred_rf)), 2),
        'MAE': round(mean_absolute_error(y_test, pred_rf), 2),
        'R2': round(r2_score(y_test, pred_rf), 4),
    })

    xgb = XGBRegressor(n_estimators=200, learning_rate=0.1, max_depth=3, random_state=42, n_jobs=2)
    xgb.fit(X_train, y_train)
    pred_xgb = xgb.predict(X_test)
    results.append({
        'comparison': name,
        'model': 'XGBoost',
        'RMSE': round(np.sqrt(mean_squared_error(y_test, pred_xgb)), 2),
        'MAE': round(mean_absolute_error(y_test, pred_xgb), 2),
        'R2': round(r2_score(y_test, pred_xgb), 4),
    })

# Display coefficients for each ACLED feature to determine significance
lr_ext, ext_feature_order = fitted_lr['extended_acled']
acled_coef_df = pd.DataFrame({
    'feature'    : ext_feature_order,
    'coefficient': lr_ext.coef_,
})
acled_coef_df = acled_coef_df[acled_coef_df['feature'].isin(ACLED_FEATURES)]
acled_coef_df = acled_coef_df.reindex(
    acled_coef_df['coefficient'].abs().sort_values(ascending=False).index
)
print(acled_coef_df.to_string(index=False))

results_df = pd.DataFrame(results)
results_df


Baseline rows           : 60
Rows after merge/filter : 59
Date range              : 2020-02-01 to 2024-12-01
Train rows              : 23
Test rows               : 36
Features                : 26

Design matrix rank: 13 features of the 26 columns
are collinear (highly correlated, i.e., duplicate information)!

                 feature   coefficient
            acled_states  2.674800e+06
     acled_events_roll_3 -8.918275e+05
      acled_fatal_events  6.993276e+05
acled_fatal_events_lag_1 -6.138432e+05
  acled_total_fatalities -4.987396e+05
      acled_events_lag_1  4.083184e+05
       acled_event_types -3.955064e+05
acled_high_activity_flag -2.996342e+05
acled_fatal_events_lag_2  2.725477e+05
 acled_fatalities_roll_3  2.367873e+05
      acled_events_lag_2 -2.172612e+05
         acled_any_event  0.000000e+00
   acled_any_fatal_event  0.000000e+00


,comparison,model,RMSE,MAE,R2
0,baseline_acled,Linear Regression,3267496.43,3083663.46,-7.4773
1,baseline_acled,Random Forest,3153694.31,2963952.95,-6.8971
2,baseline_acled,XGBoost,3922408.34,3662260.50,-11.2161
3,extended_acled,Linear Regression,3454310.90,3145490.10,-8.4743
4,extended_acled,Random Forest,3681136.48,3463448.37,-9.7594
5,extended_acled,XGBoost,3304305.80,3130983.12,-7.6693


These results don't support the extended ACLED feature set. Adding the extra columns made two of our
three models worse: linear regression went from 3.27M to 3.45M RMSE and random forest went from 3.15M
to 3.68M, with only XGBoost improving. The random forest pair says it best, since it's the same model
with the same hyperparameters and thirteen extra features, and RMSE still gets about 17% worse. That's
exactly what the rank check above is telling us (26 features, 23 training rows, rank 13). It's also
worth noting that every R2 here is strongly negative, which means all six models do worse than simply
predicting the test average — that's our global dropna() at work, collapsing us to Feb 2020 through Dec
2023 so we train on the pandemic collapse (~1.7M mean arrivals) and test on the recovery (~5.7M). We
shouldn't read the coefficient table as a ranking of what matters, so we follow this up properly in
Additional_Modeling.ipynb.

In [16]:
peter_mask = df["date"] < split_date

X_train_peter = df.loc[peter_mask, FEATURES]
X_test_peter = df.loc[~peter_mask, FEATURES]
y_train_peter = df.loc[peter_mask, LABEL]
y_test_peter = df.loc[~peter_mask, LABEL]

def evaluate_model(name, model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    mae = mean_absolute_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)

    print(f"{name}")
    print(f"RMSE : {rmse:,.0f}")
    print(f"MAE  : {mae:,.0f}")
    print(f"R²   : {r2:.4f}")
    print()

    return {
        "model": name,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "predictions": predictions
    }

peter_results = []

# Random forest does not require feature scaling.
rf_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=3,
    min_samples_leaf=2,
    max_features=0.8,
    random_state=42,
    n_jobs=-1
)

peter_results.append(
    evaluate_model(
        "Random Forest",
        rf_model,
        X_train_peter,
        y_train_peter,
        X_test_peter,
        y_test_peter
    )
)

# XGBoost also does not require feature scaling.
xgb_model = XGBRegressor(
    n_estimators=250,
    max_depth=2,
    learning_rate=0.03,
    min_child_weight=2,
    subsample=0.8,
    colsample_bytree=0.9,
    reg_lambda=5,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=1
)

peter_results.append(
    evaluate_model(
        "XGBoost",
        xgb_model,
        X_train_peter,
        y_train_peter,
        X_test_peter,
        y_test_peter
    )
)

peter_results_df = pd.DataFrame(peter_results).drop(columns=["predictions"])
peter_results_df.sort_values("RMSE")

Random Forest
RMSE : 3,238,580
MAE  : 3,036,478
R²   : -7.3279

XGBoost
RMSE : 3,569,751
MAE  : 3,386,271
R²   : -9.1182



,model,RMSE,MAE,R2
0,Random Forest,3.238580e+06,3.036478e+06,-7.327892
1,XGBoost,3.569751e+06,3.386271e+06,-9.118163


# Hope Modeling Section